# VLM Integrated Gradients - Google Colab

This notebook uses Integrated Gradients to explain Vision-Language Model predictions.

**Requirements:** Google Colab with GPU runtime (Runtime > Change runtime type > GPU)

## Step 1: Clone Repository and Install Dependencies

In [ ]:
# Clone your GitHub repository
!git clone https://github.com/YOUR_USERNAME/YOUR_REPO_NAME.git
%cd YOUR_REPO_NAME

In [ ]:
# Install numpy first (required specific version)
!pip uninstall -y numpy
!pip install numpy==1.26.4

print("\n⚠️  Please restart runtime: Runtime > Restart runtime")
print("After restart, skip this cell and run the next one.")

In [ ]:
# After restart, install other dependencies
!pip install -q -r requirements.txt

## Step 2: Import Libraries and Load Model

In [ ]:
# Import the modules
import sys
sys.path.insert(0, './src')

from src import load_model, vqa_interpret, temporal_vqa_interpret, temporal_query_vqa_interpret, conditional_query_vqa_interpret, text_vqa_interpret
import torch
import gc

print("✓ Imports successful")

In [ ]:
# Load the model
model, processor = load_model()

# Check GPU status
if torch.cuda.is_available():
    print(f"\nGPU: {torch.cuda.get_device_name(0)}")
    print(f"Total memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("\n⚠️  Warning: GPU not available. Please enable GPU in Runtime > Change runtime type")

## Step 3: Upload Image and Run Analysis

In [ ]:
# Clear GPU memory before analysis
gc.collect()
torch.cuda.empty_cache()

print("✓ Memory cleared and ready")

In [ ]:
# Upload your image
from google.colab import files

print("Click 'Choose Files' to upload an image...")
uploaded = files.upload()

if uploaded:
    image_path = list(uploaded.keys())[0]
    print(f"\n✓ Uploaded: {image_path}")
else:
    print("No file uploaded.")

In [ ]:
# Define your questions
questions = [
    "What is in this image?",
    "What color is the main object?"
]

# Run the analysis
vqa_interpret(
    image_path=image_path,
    questions=questions,
    model=model,
    processor=processor,
    show_top_k=10
)

## Optional: Run Another Analysis

You can run the above two cells again to analyze a different image with different questions.

## Step 4a: Full Video Analysis (Optional)

Upload and analyze an entire video file with temporal processing.

In [ ]:
# Clear GPU memory before video analysis
gc.collect()
torch.cuda.empty_cache()

print("✓ Memory cleared and ready for video analysis")

In [ ]:
# Upload your video
from google.colab import files

print("Click 'Choose Files' to upload a video...")
uploaded_video = files.upload()

if uploaded_video:
    video_path = list(uploaded_video.keys())[0]
    print(f"\n✓ Uploaded: {video_path}")
else:
    print("No video uploaded.")

In [ ]:
# Define questions for video analysis
video_questions = [
    "What is happening?",
    "What object is visible?"
]

# Run temporal analysis
# Parameters:
# - fps_sample: How many frames per second to sample (1 = 1 frame/sec, 0.5 = 1 frame/2 sec)
# - max_frames: Maximum frames to process (None = all sampled frames)
# - show_frame_visualizations: Show heatmap for each frame (slow, use False for long videos)
# - show_timeline: Show timeline graphs of predictions over time
# - save_results: Save reports and visualizations to files
# - include_text_attribution: Compute image vs text attribution per frame

results = temporal_vqa_interpret(
    video_path=video_path,
    questions=video_questions,
    model=model,
    processor=processor,
    fps_sample=1,  # Sample 1 frame per second
    max_frames=20,  # Process max 20 frames (adjust based on video length)
    show_frame_visualizations=False,  # Set True to see individual frame heatmaps
    show_timeline=True,  # Show timeline graphs
    save_results=True,  # Save reports to 'video_analysis' folder
    include_text_attribution=True  # Enable text vs image attribution analysis
)

In [ ]:
# Define timestamp-specific questions
# The system automatically parses temporal references like "at 4 seconds", "first 10 seconds", etc.
temporal_questions = [
    "What is happening at 4 seconds?",
    "What's in the first 10 seconds?",
    "What occurs between 5 and 8 seconds?",
    "What's at the beginning of the video?",
]

# Run temporal query analysis - analyzes ONLY relevant frames (much faster!)
# Parameters:
# - fps_sample: How many frames per second to sample (1 = 1 frame/sec)
# - aggregation_method: How to combine multi-frame results
#     - 'most_confident': Use frame with highest confidence (default)
#     - 'consensus': Use most common prediction across frames
#     - 'average': Average attributions and confidences
# - show_visualizations: Show attribution heatmaps
# - save_results: Save reports and visualizations to files
# - include_text_attribution: Compute text token importance

query_results = temporal_query_vqa_interpret(
    video_path=video_path,
    temporal_questions=temporal_questions,
    model=model,
    processor=processor,
    fps_sample=1,  # Sample 1 frame per second
    aggregation_method='most_confident',  # Options: 'most_confident', 'consensus', 'average'
    show_visualizations=True,  # Show attribution heatmaps
    save_results=True,  # Save to 'temporal_query_analysis' folder
    include_text_attribution=False  # Set True to analyze question token importance
)

# Access results
for question, result in query_results.items():
    print(f"\nQ: {question}")
    print(f"Time: {result['temporal_info']['start_sec']:.1f}s - {result['temporal_info']['end_sec']:.1f}s")
    print(f"A: {result['prediction']}")
    print(f"Confidence: {result['confidence']:.2%}")

In [ ]:
# Clear GPU memory before temporal query analysis
gc.collect()
torch.cuda.empty_cache()

print("✓ Memory cleared and ready for temporal query analysis")

## Step 4b: Temporal Query Analysis (NEW - Timestamp-Specific Questions)

Ask questions about specific timestamps in your video. Much faster than analyzing the entire video!

In [ ]:
# Define conditional questions - LLM will intelligently parse these!
# The system automatically:
# 1. Identifies the frame condition (e.g., "when there is a dog")
# 2. Scans all frames to find matches
# 3. Answers the question ONLY on matching frames

conditional_questions = [
    # Conditional queries - two-stage analysis
    "When there is a dog in the frame, what color is the dog?",
    "Where the person is running, what are they wearing?",
    "At frames with a car, what color is it?",
    
    # Timestamp queries - still supported
    "What happens at 5 seconds?",
    
    # Simple queries - analyzes all frames
    "What objects appear most often?",
]

# Run conditional query analysis
# The LLM will:
# - Parse each question to understand the condition and the actual question
# - For conditional questions: scan all frames, find matches, then answer
# - For timestamp/simple questions: use appropriate frame selection

# Parameters:
# - fps_sample: How many frames per second to sample (1 = 1 frame/sec)
# - confidence_threshold: Minimum confidence for frame matching (0.5 = balanced)
#     Higher (0.6-0.7) = stricter matching, fewer frames
#     Lower (0.3-0.4) = more lenient, more frames
# - aggregation_method: How to combine multi-frame results
#     'most_confident': Use frame with highest confidence (default)
#     'consensus': Use most common prediction (best for many matches)
#     'average': Average attributions and confidences
# - show_visualizations: Show attribution heatmaps
# - save_results: Save detailed reports

conditional_results = conditional_query_vqa_interpret(
    video_path=video_path,
    questions=conditional_questions,
    model=model,
    processor=processor,
    fps_sample=1,  # Sample 1 frame per second
    confidence_threshold=0.5,  # Balanced threshold (0.3-0.7 range)
    aggregation_method='most_confident',  # Options: 'most_confident', 'consensus', 'average'
    show_visualizations=True,
    save_results=True,  # Saves to 'conditional_query_analysis' folder
    include_text_attribution=False
)

# Display summary of results
print("\n" + "="*70)
print("CONDITIONAL QUERY RESULTS SUMMARY")
print("="*70)

for question, result in conditional_results.items():
    print(f"\n📝 Q: {question}")
    
    if 'error' in result:
        print(f"   ❌ Error: {result['error']}")
        continue
    
    # Show how the LLM parsed the question
    parsed = result['query_info']['parsed']
    print(f"   🔍 Type: {parsed['type']}")
    
    if parsed['type'] == 'conditional':
        print(f"   📋 Condition: {parsed['frame_condition']}")
        print(f"   🎯 Frames matched: {result['query_info']['num_matching_frames']}/{result['query_info']['total_frames']}")
        print(f"   📌 Frame indices: {result['query_info']['frame_indices']}")
    
    # Show the answer
    print(f"   💡 Answer: {result['prediction']}")
    print(f"   📊 Confidence: {result['confidence']:.2%}")

In [ ]:
# Clear GPU memory before conditional query analysis
gc.collect()
torch.cuda.empty_cache()

print("✓ Memory cleared and ready for conditional query analysis")

## Step 4c: Conditional Query Analysis (NEW - LLM-Based Smart Frame Selection)

Ask complex conditional questions! The LLM automatically identifies frames matching your condition, then answers the question.

Examples:
- "When there is a dog in the frame, what color is the dog?"
- "Where the person is running, what are they wearing?"
- "At frames with a car, is it red or blue?"

## Step 5: Text Attribution Analysis (Optional)

Analyze which words in your question influence the prediction, and compare image vs text contribution.

In [ ]:
# Clear GPU memory before text attribution analysis
gc.collect()
torch.cuda.empty_cache()

print("✓ Memory cleared and ready for text attribution analysis")

In [ ]:
# Define questions for text attribution
text_questions = [
    "What color is the cat?",
    "Where is the cat?"
]

text_results = text_vqa_interpret(
    image_path=image_path,
    questions=text_questions,
    model=model,
    processor=processor,
    mode='both',
    n_steps=10,
    show_visualizations=True
)